# 추상화 인터페이스

In [3]:
from abc import ABC, abstractmethod
class BaseGraph(ABC):
    @abstractmethod
    def add_node(self, node_key):
        pass
    
    @abstractmethod
    def delete_node(self, node_key):
        pass
    
    @abstractmethod
    def add_edge(self, start_node:str, end_node:str):
        pass
    
    @abstractmethod
    def delete_edge(self, start_node:str, end_node:str):
        pass

# 인접 리스트를 이용한 Graph

In [4]:
# 인접 리스트
class AdjencyListGraph(BaseGraph):
    def __init__(self, graph:dict[str, list[str]]):
        self.graph = graph

    def add_node(self, node_key):
        """Worst: O(N)"""
        if node_key not in self.graph:
            self.graph[node_key] = []
            print(f"Graph: {self.graph}")
            return True
        print(f"{node_key} already exists.")
        return False
    
    def delete_node(self, node_key):
        """Worst: O(N)"""
        if node_key not in self.graph: # node_key가 없으면 삭제할 게 없는 것
            print(f"{node_key} doesn't exist.")
            return False
        for key in self.graph:
            if node_key in self.graph[key]:
                self.graph[key].remove(node_key)
        del self.graph[node_key]
        print(f"{node_key} get deleted")
        print(f"Graph after Deletion: {self.graph}")
        return True
    
    def add_edge(self, start_node:str, end_node:str):
        """Worst: O(k log k)"""
        if start_node not in self.graph:
            print(f"{start_node}가 그래프에 존재하지 않습니다.")
            return False
        
        if end_node not in self.graph :
            print(f"{end_node}가 그래프에 존재하지 않습니다.")
            return False
        
        if end_node in self.graph[start_node]:
            print(f"이미 {start_node} - {end_node} 간선이 존재합니다.")
            return False
        
        # 연결
        self.graph[start_node].append(end_node)
        self.graph[end_node].append(start_node)
        
        self.graph[start_node].sort()
        self.graph[end_node].sort()
        print(f"{start_node} and {end_node} get linked.")
        print(f"Graph after linking: {self.graph}")
        return True
        
    def delete_edge(self, start_node:str, end_node:str):
        """Worst: O(N)"""
        
        if start_node not in self.graph:
            print(f"{start_node}가 그래프에 없습니다.")
            return False
        
        if end_node not in self.graph:
            print(f"{end_node}가 그래프에 없습니다.")
            return False
    
        if end_node in self.graph[start_node] and start_node in self.graph[end_node]: # 서로 연결되어 있어야 삭제가 가능하므로 확인
            self.graph[start_node].remove(end_node)
            self.graph[end_node].remove(start_node)
            print(f"{start_node} - {end_node}가 삭제 되었습니다.")
            print(f"Graph after edge deletion: {self.graph}")
            return True
        else:
            print(f"{start_node} | {end_node} 노드는 존재하지만, 연결 정보가 없습니다.")
            return False 
        
graph = AdjencyListGraph({"A":["B","C"],
               "B":["A","C"],
               "C":["A","B","C"],
               "D":["C"]})

graph.add_node("Q")
graph.delete_node("Q")
graph.add_edge("A", "D")
graph.delete_edge("Q", "Q")

Graph: {'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B', 'C'], 'D': ['C'], 'Q': []}
Q get deleted
Graph after Deletion: {'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B', 'C'], 'D': ['C']}
A and D get linked.
Graph after linking: {'A': ['B', 'C', 'D'], 'B': ['A', 'C'], 'C': ['A', 'B', 'C'], 'D': ['A', 'C']}
Q가 그래프에 없습니다.


False

# 인접행렬을 이용한 Graph

In [23]:
class AdjacencyMatrixGraph(BaseGraph):
    def __init__(self, matrix: list[list], vertex_name_list: list):
        self.matrix = matrix       # 2차원 인접 행렬
        if len(self.matrix) != len(vertex_name_list):
            raise BaseException("정점 이름 목록과 인접행렬의 크기가 다릅니다.")
        self.vertex_to_idx = {name:idx for idx, name in enumerate(vertex_name_list)}   # 정점 이름 -> 행렬 인덱스 매핑
        self.idx_to_vertex = {idx:name for idx, name in enumerate(vertex_name_list)}   # 행렬 인덱스 -> 정점 이름 매핑 (삭제 시 갱신용)
        self.length = len(matrix)
    
    def add_node(self, node_key):
        """Worst: O(N)"""
        if node_key not in self.vertex_to_idx:
            self.vertex_to_idx[node_key] = self.length
            self.idx_to_vertex[self.length] = node_key
            self.length += 1
            for row in self.matrix:
                row.append(0)
            self.matrix.append([0] * self.length)
            return True
        print(f"{node_key} already exists.")
        return False
    
    def delete_node(self, node_key):
        """Worst O(V)"""
        if node_key not in self.vertex_to_idx:
            print(f"해당 {node_key} 정점이 없습니다.")
            return False
        # 노드가 있다는 것
        delete_idx = self.vertex_to_idx[node_key]
        for row in self.matrix:
            row.pop(delete_idx)
        self.matrix.pop(delete_idx)        
        print(f"해당 {node_key} 정점이 삭제되었습니다.")
        print(f"Graph: {self.matrix}")
        return True
    
    def add_edge(self, start_node:str, end_node:str):
        if start_node not in self.vertex_to_idx:
            print(f"해당 {start_node} 정점이 없습니다.")
            return False
        
        if end_node not in self.vertex_to_idx:
            print(f"해당 {end_node} 정점이 없습니다.")
            return False
            
        start_idx = self.vertex_to_idx[start_node]
        end_idx = self.vertex_to_idx[end_node]
        self.matrix[start_idx][end_idx] = 1
        self.matrix[end_idx][start_idx] = 1
        print(f"해당 {start_node} - {end_node} 정점이 연결되었습니다.")
        print(f"Graph: {self.matrix}")
        return True
    
    def delete_edge(self, start_node:str, end_node:str):
        if start_node not in self.vertex_to_idx:
            print(f"해당 {start_node} 정점이 없습니다.")
            return False
        
        if end_node not in self.vertex_to_idx:
            print(f"해당 {end_node} 정점이 없습니다.")
            return False
            
        start_idx = self.vertex_to_idx[start_node]
        end_idx = self.vertex_to_idx[end_node]
        self.matrix[start_idx][end_idx] = 0
        self.matrix[end_idx][start_idx] = 0
        print(f"해당 {start_node} - {end_node} 정점이 연결 해제되었습니다.")
        print(f"Graph: {self.matrix}")
        return True
    
    def __repr__(self):
        return f"{self.matrix}"
    
graph = AdjacencyMatrixGraph([[0,1,1,0],
                              [1,0,1,0],
                              [1,1,0,1],
                              [0,0,1,0]],
                             ["A","B","C","D"])

graph.add_edge("A","D")

해당 A - D 정점이 연결되었습니다.
Graph: [[0, 1, 1, 1], [1, 0, 1, 0], [1, 1, 0, 1], [1, 0, 1, 0]]


True